In [14]:
from ultralytics import YOLO
from manga_ocr import MangaOcr
import cv2
from PIL import Image
from transformers import pipeline
import numpy as np

c:\Users\majd\Desktop\pfe project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:

import os
from ollama import Client

# 1. Set your API key (from https://ollama.com/settings/keys)
#    - Create one if you don't have it (free tier exists, paid for heavy use)
os.environ["OLLAMA_API_KEY"] = "59829075036e4cb5bd4f3ef53b27fbcd.fpdV5oI5BTenSXgHxepcE_qL"  # ← your real key here

# 2. Create client with correct cloud endpoint
client = Client(
    host="https://ollama.com",  # ← This is the required /api path
    headers={
        'Authorization': f'Bearer {os.environ["OLLAMA_API_KEY"]}'
    }
)
llm_model_name = "gpt-oss:120b-cloud"

In [16]:
panel_detector = YOLO("models/mosesb.pt")

In [17]:
bubble_detector = YOLO("models/comic-speech-bubble-detector.pt")

In [18]:
ocr = MangaOcr()

2026-02-21 13:28:13.111 | INFO     | manga_ocr.ocr:__init__:16 - Loading OCR model from kha-white/manga-ocr-base
Loading weights: 100%|██████████| 264/264 [00:00<00:00, 496.23it/s, Materializing param=encoder.pooler.dense.weight]                                       
The tied weights mapping and config for this model specifies to tie decoder.bert.embeddings.word_embeddings.weight to decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie decoder.cls.predictions.bias to decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
MangaOcrModel LOAD REPORT from: kha-white/manga-ocr-base
Key                                  | Status     |  | 
------------------

In [19]:
image_file = "img/005.jpg"

In [20]:
img = cv2.imread(image_file)

In [21]:
panel_results = panel_detector(img, conf=0.2)


0: 640x480 6 Comic Panels, 1847.7ms
Speed: 167.5ms preprocess, 1847.7ms inference, 52.6ms postprocess per image at shape (1, 3, 640, 480)


In [22]:
anotation = panel_results[0].plot()
img_results ="panel_detection.jpg"
cv2.imwrite(img_results,anotation)

True

In [23]:
number = 1
panels = []
for box in panel_results[0].boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    panels.append({
        "x1": x1, "y1": y1, "x2": x2, "y2": y2,
        "crop": img[y1:y2, x1:x2]
    }
    )
    crop = img[y1:y2, x1:x2]
    cv2.imwrite(f"crop{number}.jpg",crop)
    number +=1

In [24]:
h, w = img.shape[:2]

# ── Helper: Compute IoU between two boxes [x1,y1,x2,y2] ───────────────────────
def compute_iou(boxA, boxB):
    # Intersection coordinates
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    
    inter_area = max(0, xB - xA) * max(0, yB - yA)
    
    # Areas
    boxA_area = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxB_area = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    
    if boxA_area + boxB_area - inter_area == 0:
        return 0.0
    return inter_area / float(boxA_area + boxB_area - inter_area)

# ── Step 1: Detect panels on full page ───────────────────────────────────────
print("Detecting panels...")
panel_results = panel_detector(img, conf=0.25)
panels = []
for box in panel_results[0].boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    conf = float(box.conf)
    if conf >= 0.25:
        panels.append({
            "box": [x1, y1, x2, y2],   # for IoU
            "crop": img[y1:y2, x1:x2],
            "conf": conf
        })
print(f"Found {len(panels)} panels")

# ── Step 2: Detect bubbles on FULL page ──────────────────────────────────────
print("Detecting bubbles on full page...")
bubble_results = bubble_detector(img, conf=0.3)
bubbles = []
for box in bubble_results[0].boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    conf = float(box.conf)
    if conf >= 0.25:
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        bubble_crop = img[y1:y2, x1:x2]
        bubble_pil = Image.fromarray(cv2.cvtColor(bubble_crop, cv2.COLOR_BGR2RGB))
        
        bubbles.append({
            "box": [x1, y1, x2, y2],
            "cx": cx,
            "cy": cy,
            "crop_pil": bubble_pil,
            "conf": conf
        })
print(f"Found {len(bubbles)} bubbles")

# ── Step 3: Assign bubbles to panels using best IoU ──────────────────────────
panel_to_bubbles = {i: [] for i in range(len(panels))}

for bubble in bubbles:
    best_iou = 0.0
    best_panel_idx = -1
    
    for idx, panel in enumerate(panels):
        iou = compute_iou(bubble["box"], panel["box"])
        if iou > best_iou:
            best_iou = iou
            best_panel_idx = idx
    
    # Fallback: if no good IoU, assign by center point
    if best_iou < 0.1:  # adjust threshold if needed
        bubble_center = (bubble["cx"], bubble["cy"])
        for idx, panel in enumerate(panels):
            bx1, by1, bx2, by2 = panel["box"]
            if bx1 <= bubble["cx"] <= bx2 and by1 <= bubble["cy"] <= by2:
                best_panel_idx = idx
                break
    
    if best_panel_idx != -1:
        panel_to_bubbles[best_panel_idx].append(bubble)
    else:
        print(f"Warning: Bubble {bubble['box']} not assigned to any panel")

# ── Step 4: Sort panels in reading order ─────────────────────────────────────
def get_center(item):
    box = item["box"] if "box" in item else [item["x1"], item["y1"], item["x2"], item["y2"]]
    return ((box[0] + box[2]) / 2, (box[1] + box[3]) / 2)

# Sort: top-to-bottom (cy asc), then right-to-left (-cx desc)
sorted_panel_indices = sorted(
    range(len(panels)),
    key=lambda i: (get_center(panels[i])[1], -get_center(panels[i])[0])
)

print(f"Sorted {len(sorted_panel_indices)} panels in manga reading order")

# ── Step 5: For each sorted panel → sort its bubbles ─────────────────────────
all_ordered_bubbles = []

for panel_idx in sorted_panel_indices:
    panel_bubbles = panel_to_bubbles[panel_idx]
    
    # Sort bubbles inside panel: top-to-bottom, right-to-left
    sorted_bubbles = sorted(
        panel_bubbles,
        key=lambda b: (b["cy"], -b["cx"])
    )
    
    all_ordered_bubbles.extend(sorted_bubbles)

print(f"Final ordered bubbles: {len(all_ordered_bubbles)}")

Detecting panels...

0: 640x480 6 Comic Panels, 1429.8ms
Speed: 13.3ms preprocess, 1429.8ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 480)
Found 6 panels
Detecting bubbles on full page...

0: 1024x736 9 text_bubbles, 1205.3ms
Speed: 15.9ms preprocess, 1205.3ms inference, 14.8ms postprocess per image at shape (1, 3, 1024, 736)
Found 9 bubbles
Sorted 6 panels in manga reading order
Final ordered bubbles: 9


In [25]:
from transformers import MarianMTModel, MarianTokenizer

# Load the Japanese → Arabic model
model_name = "Helsinki-NLP/opus-mt-ja-ar"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

c:\Users\majd\Desktop\pfe project\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 487.41it/s, Materializing param=model.shared.weight]                                  
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [26]:
def translate_ja_to_ar(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        src_lang="jpn_Jpan"
    )

    arb_id = tokenizer.convert_tokens_to_ids("fra_Latn")

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=arb_id,
        max_length=256
    )

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

print(translate_ja_to_ar("私は学生です。"))

c.arabic c.arabicrlm; أنا طالب. /c.arabic c.arabicrlm;


In [ ]:
# pip install --upgrade ollama   # Run this once if needed

# def lmm_translation(japanese):
# 3. Example chat (streaming)


In [38]:
bubble_number = 1
for bubble in all_ordered_bubbles:
    japanese = ocr(bubble["crop_pil"]).strip()
    print(f"Bubble {bubble_number}:")
    print(f"Japanese: {japanese}")
    messages = [
       {
        'role': 'user',
        'content': f"""أنت مترجم محترف متخصص في ترجمة المانجا من اليابانية إلى العربية الفصحى (MSA).
           القواعد الصارمة: 
           - استخدم العربية الفصحى فقط، بدون أي لهجة عامية.
           - حافظ على الطابع الرسمي والأدبي.
           - احتفظ بالعواطف، التعجبات، والأسلوب المانجا.
           - أخرج الترجمة العربية الفصحى فقط.
          النص الياباني:
          {japanese}"""

    }
      
      
      
      ]

# Use a real cloud model tag (examples below)
  # or "qwen2.5:72b-instruct:cloud", "llama3.1:405b:cloud", etc.
# Check available: curl https://ollama.com/api/tags -H "Authorization: Bearer YOUR_KEY"

    print(f"Streaming response from {llm_model_name}:\n")

    for part in client.chat(llm_model_name, messages=messages, stream=True):
     print(part['message']['content'], end='', flush=True)  # ← Note: part is dict, access ['message']['content']

    
    print("─" * 50)
    bubble_number += 1
    # try:
    # #  Tokenize input (MarianMT expects the text directly)
    #  inputs = tokenizer(japanese, return_tensors="pt", padding=True)
    
    # #  Generate translation
    #  translated_ids = model.generate(
    #     **inputs,
    #     max_length=200,
    #     num_beams=4,          # optional: improves quality a bit
    #     early_stopping=True
    # )
    
    #  arabic = tokenizer.decode(translated_ids[0], skip_special_tokens=True).strip()
    #  print(arabic)
    # except Exception as e:
    #  print(f"Bubble {bubble_number}: Translation failed → {e}")
    #  arabic = "[translation error]"

Bubble 1:
Japanese: ちっくしょ〜
Streaming response from gpt-oss:120b-cloud:

تبًا!──────────────────────────────────────────────────
Bubble 2:
Japanese: 手伝わんなら推薦はやらん
Streaming response from gpt-oss:120b-cloud:

إن لم تُساعد، فلن أُقدِّم التوصية.──────────────────────────────────────────────────
Bubble 3:
Japanese: 脅迫じゃねーかあのパワハラ教師！！
Streaming response from gpt-oss:120b-cloud:

أليس هذا تهديدًا من ذلك المعلم المتسلط؟!──────────────────────────────────────────────────
Bubble 4:
Japanese: なぜ俺がこんなことを．．．
Streaming response from gpt-oss:120b-cloud:

لماذا أفعل هذا…──────────────────────────────────────────────────
Bubble 5:
Japanese: ．．．天文部
Streaming response from gpt-oss:120b-cloud:

…نادي الفلك──────────────────────────────────────────────────
Bubble 6:
Japanese: ここか
Streaming response from gpt-oss:120b-cloud:

هنا؟──────────────────────────────────────────────────
Bubble 7:
Japanese: うぉぉぉおぉおおおぉ！？
Streaming response from gpt-oss:120b-cloud:

ووووووووووه!؟─────────────────────────────────────────

In [40]:
# ==============================================
# Japanese → English → Arabic via two Opus-MT models
# Requirements: pip install transformers torch sentencepiece
# ==============================================

from transformers import MarianMTModel, MarianTokenizer
import torch

# ── Load models & tokenizers once (at startup) ────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

# Model 1: Japanese → English
model_ja_en_name = "Helsinki-NLP/opus-mt-ja-en"
tokenizer_ja_en = MarianTokenizer.from_pretrained(model_ja_en_name)
model_ja_en = MarianMTModel.from_pretrained(model_ja_en_name).to(device)

# Model 2: English → Arabic
model_en_ar_name = "Helsinki-NLP/opus-mt-en-ar"
tokenizer_en_ar = MarianTokenizer.from_pretrained(model_en_ar_name)
model_en_ar = MarianMTModel.from_pretrained(model_en_ar_name).to(device)

print("Models loaded successfully!")

# ── Translation function ──────────────────────────────────────────────────────
def translate_ja_to_ar_via_en(japanese_text: str, max_length: int = 200) -> str:
    """
    Translate Japanese text to Arabic by pivoting through English.
    
    Args:
        japanese_text: Single Japanese string (or list of strings)
        max_length: Max tokens in generation
    
    Returns:
        Translated Arabic string
    """
    # Step 1: Japanese → English
    inputs_ja = tokenizer_ja_en(
        japanese_text,
        return_tensors="pt",
        padding=True
    ).to(device)
    
    translated_en_ids = model_ja_en.generate(
        **inputs_ja,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )
    
    english_text = tokenizer_ja_en.batch_decode(
        translated_en_ids,
        skip_special_tokens=True
    )[0].strip()
    
    # Step 2: English → Arabic
    inputs_en = tokenizer_en_ar(
        english_text,
        return_tensors="pt",
        padding=True
    ).to(device)
    
    translated_ar_ids = model_en_ar.generate(
        **inputs_en,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )
    
    arabic_text = tokenizer_en_ar.batch_decode(
        translated_ar_ids,
        skip_special_tokens=True
    )[0].strip()
    
    return arabic_text , english_text


# ── Example usage ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    test_sentences = [
        "ちっくしょ〜",
        "脅迫じゃねーかあのパワハラ教師！！",
        "うぉぉぉおぉおおおぉ！？",
        "ん？",
        "失礼しまーす"
    ]
    
    print("Japanese → Arabic via English pivot:\n")
    for i, ja in enumerate(test_sentences, 1):
        ar = translate_ja_to_ar_via_en(ja)
        print(f"Bubble {i}:")

        print(f"Japanese: {ja}")
        print(f"Arabic   : {ar}")
        print("─" * 50)

c:\Users\majd\Desktop\pfe project\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 379.18it/s, Materializing param=model.shared.weight]                                  
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 388.95it/s, Materializing 

Models loaded successfully!
Japanese → Arabic via English pivot:

Bubble 1:
Japanese: ちっくしょ〜
Arabic   : ('أوه، تغوّط.', 'Oh, shit.')
──────────────────────────────────────────────────
Bubble 2:
Japanese: 脅迫じゃねーかあのパワハラ教師！！
Arabic   : ('هذا ليس تهديداً هذا معلم', "That's not a threat! That's a teacher!")
──────────────────────────────────────────────────
Bubble 3:
Japanese: うぉぉぉおぉおおおぉ！？
Arabic   : ('مهلاً، مهلاً، مهلاً، مهلاً!', 'Whoa, whoa, whoa!')
──────────────────────────────────────────────────
Bubble 4:
Japanese: ん？
Arabic   : ('ماذا؟', 'Hmm?')
──────────────────────────────────────────────────
Bubble 5:
Japanese: 失礼しまーす
Arabic   : ('-التمست مني.', 'Excuse me.')
──────────────────────────────────────────────────
